end

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp
from tqdm import tqdm
import textwrap
import logging

# ==================== 1. 配置路径 (适配小鼠数据) ====================
# 输入: 上一步生成的小鼠聚类结果目录
INPUT_CLUSTER_DIR = '/mnt/e/2-8.3-shanda/1-feature/10-2-0-Gene_Trajectory_Clusters_Annotated/'
# 输出: 本次 GO 富集分析的保存目录
OUTPUT_GO_DIR = '/mnt/e/2-8.3-shanda/1-feature/1-figure/0-3-result-4-nonlinear/2-cluster-GO'

# ==================== 2. 参数设置 ====================
# 设置要提取的 Top N 显著通路用于画图
TOP_N_TERMS = 8  
# 选择富集数据库
ENRICH_DB = 'GO_Biological_Process_2023' 

os.makedirs(OUTPUT_GO_DIR, exist_ok=True)

# 设置顶级期刊风格的全局绘图参数
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
sns.set_theme(style="ticks", rc={'axes.edgecolor': 'black'})

def normalize_gene_name_mouse(name: str) -> str:
    """🐭 小鼠专属：将基因名规范为首字母大写、其余小写 (如 GAPDH -> Gapdh)"""
    if not isinstance(name, str) or not name.strip():
        return str(name)
    name = name.strip()
    return name[0].upper() + name[1:].lower()

print(f"🚀 开始进行小鼠组织的 GO 富集分析与气泡图绘制 (紧凑布局版)...")

# 寻找所有上一步生成的聚类结果文件
cluster_files = [f for f in os.listdir(INPUT_CLUSTER_DIR) if f.endswith('_Cluster_Assignments.csv')]

for c_file in tqdm(cluster_files, desc="Processing GO Enrichment"):
    tissue_name = c_file.replace('_Cluster_Assignments.csv', '')
    print(f"\n👉 正在处理组织: {tissue_name}")
    
    # 1. 读取聚类结果
    df_clusters = pd.read_csv(os.path.join(INPUT_CLUSTER_DIR, c_file))
    df_clusters.columns = ['Gene', 'Cluster']
    
    all_enrich_results = []
    
    # 2. 对每个 Cluster 进行 GO 富集分析
    unique_clusters = sorted(df_clusters['Cluster'].unique())
    for cluster_id in unique_clusters:
        # 提取当前 Cluster 的基因列表
        raw_gene_list = df_clusters[df_clusters['Cluster'] == cluster_id]['Gene'].tolist()
        
        if len(raw_gene_list) < 5:
            print(f"  - Cluster {cluster_id + 1} 基因数太少({len(raw_gene_list)}), 跳过富集。")
            continue
            
        # 🐭 小鼠核心：转换基因名格式
        mouse_gene_list = [normalize_gene_name_mouse(g) for g in raw_gene_list]
            
        print(f"  - 正在富集 Cluster {cluster_id + 1} (包含 {len(mouse_gene_list)} 个基因)...")
        
        # ================= 核心：加入网络请求重试机制与延时 =================
        max_retries = 3
        for attempt in range(max_retries):
            try:
                # 每次请求前强制暂停 3 秒，防止被 Enrichr 服务器封禁
                time.sleep(3) 
                
                # 🐭 小鼠核心：调用 Enrichr 接口 (专门针对 Mouse)
                enr = gp.enrichr(gene_list=mouse_gene_list,
                                 gene_sets=ENRICH_DB,
                                 organism='mouse', 
                                 outdir=None,      
                                 no_plot=True)     
                
                res_df = enr.results
                # 过滤掉不显著的通路 (P-value >= 0.05)
                res_df = res_df[res_df['P-value'] < 0.05].copy()
                
                if not res_df.empty:
                    # 提取 Overlap 列中的分子 (命中该通路的基因数)
                    res_df['Count'] = res_df['Overlap'].apply(lambda x: int(x.split('/')[0]))
                    res_df['Cluster'] = f"Cluster {cluster_id + 1}"
                    
                    # 按 P-value 排序，提取最显著的 Top N
                    top_terms = res_df.sort_values('P-value').head(TOP_N_TERMS)
                    all_enrich_results.append(top_terms)
                
                break  # 成功提取，跳出重试循环
                
            except Exception as e:
                error_msg = str(e)
                if attempt < max_retries - 1:
                    print(f"    ⚠️ 网络连接异常，等待 5 秒后进行第 {attempt + 2} 次重试...")
                    time.sleep(5)
                else:
                    print(f"    ❌ Cluster {cluster_id + 1} 最终富集失败: {error_msg}")
        # ==============================================================
            
    if not all_enrich_results:
        print(f"  ⚠️ {tissue_name} 没有任何 Cluster 得到显著富集结果。")
        continue
        
    # 3. 合并所有 Cluster 的富集结果
    final_plot_df = pd.concat(all_enrich_results, ignore_index=True)
    
    # 清洗数据：去掉 GO term 名字后面的 (GO:XXXXX)
    final_plot_df['Term_Clean'] = final_plot_df['Term'].apply(lambda x: x.split(' (GO:')[0])
    # 计算 -log10(P-value) 用于颜色的深度映射
    final_plot_df['-log10(P-value)'] = -np.log10(final_plot_df['P-value'])
    
    # 保存完整的表格数据
    csv_out = os.path.join(OUTPUT_GO_DIR, f"{tissue_name}_GO_Enrichment_Summary.csv")
    final_plot_df.to_csv(csv_out, index=False)
    
    # ==================== 4. 绘制气泡图 (Bubble Plot) ====================
    # 🌟 修复处：将原本固定的宽度 8 改为动态紧凑宽度（每个 cluster 大约 0.8 inch，最少 4.5 inch 保证美观）
    dynamic_width = max(4.5, len(unique_clusters) * 0.8)
    dynamic_height = max(5, len(final_plot_df['Term_Clean'].unique()) * 0.35)
    
    plt.figure(figsize=(dynamic_width, dynamic_height))
    
    scatter = sns.scatterplot(data=final_plot_df, 
                              x='Cluster', 
                              y='Term_Clean',
                              size='Count', 
                              hue='-log10(P-value)', 
                              sizes=(50, 400), 
                              palette='viridis', 
                              edgecolor='black',
                              linewidth=0.5)
    
    # 图表细节调整
    plt.title(f"{tissue_name} - Gene Ontology Enrichment", fontsize=14, pad=15, fontweight='bold')
    plt.xlabel("")
    plt.ylabel("")
    
    # X轴标签 45 度倾斜
    plt.xticks(rotation=45, ha='right', fontsize=11)
    
    # 将图例移到图表外部
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., title_fontsize=10)
    
    # 添加背景网格
    plt.grid(True, linestyle='--', alpha=0.5, zorder=0)
    scatter.set_axisbelow(True)
    
    # 保存画布
    pdf_out = os.path.join(OUTPUT_GO_DIR, f"{tissue_name}_GO_BubblePlot.pdf")
    plt.savefig(pdf_out, bbox_inches='tight', transparent=True)
    
    # 彻底释放内存并防卡死
    plt.close()

print(f"\n✅ 完成！所有小鼠组织的 GO 富集紧凑版气泡图已保存在:\n{OUTPUT_GO_DIR}")